<a href="https://colab.research.google.com/github/sumantabhargab/SAFE5G/blob/main/notebook1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q kaggle tqdm split-folders opencv-python

In [2]:
import os

os.environ["KAGGLE_USERNAME"] = "yukohitaro"

os.environ["KAGGLE_KEY"] = "KGAT_751e60ded89b28c0b4428a2689e4bfc6"

In [3]:
!kaggle datasets download \
-d mohamedmustafa/real-life-violence-situations-dataset

Dataset URL: https://www.kaggle.com/datasets/mohamedmustafa/real-life-violence-situations-dataset
License(s): copyright-authors
100% 3.58G/3.58G [02:54<00:00, 22.1MB/s]



In [4]:
!unzip -q real-life-violence-situations-dataset.zip

In [5]:
import os

for root, dirs, files in os.walk("/content/Real Life Violence Dataset"):

    if len(files):

        print(root)

        print(files[:5])

        print()

/content/Real Life Violence Dataset/Violence
['V_279.mp4', 'V_620.mp4', 'V_399.mp4', 'V_580.mp4', 'V_909.mp4']

/content/Real Life Violence Dataset/NonViolence
['NV_395.mp4', 'NV_209.mp4', 'NV_991.mp4', 'NV_833.mp4', 'NV_325.mp4']



In [6]:
import os

violence = len(os.listdir("/content/Real Life Violence Dataset/Violence"))

normal = len(os.listdir("/content/Real Life Violence Dataset/NonViolence"))

print("Violence Videos :", violence)

print("NonViolence Videos :", normal)

Violence Videos : 1000
NonViolence Videos : 1000


In [7]:
import os

print(os.path.exists("/content/Real Life Violence Dataset"))

True


In [8]:
import os
import random
import shutil

random.seed(42)

ROOT = "/content/Real Life Violence Dataset"
OUTPUT = "/content/Safe5G_Dataset"

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

classes = ["Violence", "NonViolence"]

# --------------------------------------------------
# Create Folder Structure
# --------------------------------------------------

for split in ["train", "val", "test"]:

    for cls in classes:

        os.makedirs(
            os.path.join(OUTPUT, split, cls),
            exist_ok=True
        )

# --------------------------------------------------
# Split Videos
# --------------------------------------------------

for cls in classes:

    source_folder = os.path.join(ROOT, cls)

    videos = [
        v for v in os.listdir(source_folder)
        if v.lower().endswith((".mp4", ".avi", ".mov"))
    ]

    random.shuffle(videos)

    total = len(videos)

    train_end = int(total * TRAIN_RATIO)
    val_end = train_end + int(total * VAL_RATIO)

    train_videos = videos[:train_end]
    val_videos = videos[train_end:val_end]
    test_videos = videos[val_end:]

    splits = {
        "train": train_videos,
        "val": val_videos,
        "test": test_videos
    }

    for split_name, split_list in splits.items():

        destination = os.path.join(
            OUTPUT,
            split_name,
            cls
        )

        for video in split_list:

            shutil.copy2(

                os.path.join(source_folder, video),

                os.path.join(destination, video)

            )

    print(f"{cls}")
    print(f"Train : {len(train_videos)}")
    print(f"Val   : {len(val_videos)}")
    print(f"Test  : {len(test_videos)}")
    print("-" * 40)

print("\nDataset Split Completed Successfully.")

Violence
Train : 800
Val   : 100
Test  : 100
----------------------------------------
NonViolence
Train : 800
Val   : 100
Test  : 100
----------------------------------------

Dataset Split Completed Successfully.


In [9]:
import os

ROOT = "/content/Safe5G_Dataset"

for split in ["train", "val", "test"]:

    print(f"\n===== {split.upper()} =====")

    for cls in ["Violence", "NonViolence"]:

        folder = os.path.join(ROOT, split, cls)

        print(
            cls,
            len(os.listdir(folder))
        )


===== TRAIN =====
Violence 800
NonViolence 800

===== VAL =====
Violence 100
NonViolence 100

===== TEST =====
Violence 100
NonViolence 100


In [10]:
import os
import cv2
import json

DATASET = "/content/Safe5G_Dataset"
OUTPUT = "/content/Safe5G_Frames"

FRAME_SKIP = 10
SHARPNESS_THRESHOLD = 120

IMG_SIZE = (224,224)

metadata = {}

for split in ["train","val","test"]:

    metadata[split] = {}

    for cls in ["Violence","NonViolence"]:

        metadata[split][cls] = {}

        source = os.path.join(DATASET,split,cls)

        destination = os.path.join(OUTPUT,split,cls)

        os.makedirs(destination,exist_ok=True)

        videos = sorted(os.listdir(source))

        print(f"\nExtracting {split}/{cls}")

        for video in videos:

            video_path = os.path.join(source,video)

            cap = cv2.VideoCapture(video_path)

            frame_id = 0
            saved = 0

            while True:

                ret,frame = cap.read()

                if not ret:
                    break

                if frame_id % FRAME_SKIP == 0:

                    gray = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)

                    sharpness = cv2.Laplacian(
                        gray,
                        cv2.CV_64F
                    ).var()

                    if sharpness > SHARPNESS_THRESHOLD:

                        frame = cv2.resize(
                            frame,
                            IMG_SIZE
                        )

                        filename = f"{video[:-4]}_{saved}.jpg"

                        cv2.imwrite(

                            os.path.join(
                                destination,
                                filename
                            ),

                            frame

                        )

                        saved += 1

                frame_id += 1

            cap.release()

            metadata[split][cls][video] = saved

with open("/content/frame_metadata.json","w") as f:

    json.dump(metadata,f,indent=4)

print("\nFinished.")


Extracting train/Violence

Extracting train/NonViolence

Extracting val/Violence

Extracting val/NonViolence

Extracting test/Violence

Extracting test/NonViolence

Finished.


In [11]:
import os

ROOT="/content/Safe5G_Frames"

for split in ["train","val","test"]:

    print("\n",split.upper())

    for cls in ["Violence","NonViolence"]:

        folder=os.path.join(ROOT,split,cls)

        print(cls,len(os.listdir(folder)))


 TRAIN
Violence 7083
NonViolence 9800

 VAL
Violence 822
NonViolence 1152

 TEST
Violence 843
NonViolence 1165


In [12]:
import os

ROOT = "/content/Safe5G_Frames"

total = 0

for split in ["train", "val", "test"]:

    print("\n" + "="*40)
    print(split.upper())
    print("="*40)

    for cls in ["Violence", "NonViolence"]:

        folder = os.path.join(ROOT, split, cls)

        count = len(os.listdir(folder))

        total += count

        print(f"{cls:<15}: {count}")

print("\nTotal Images:", total)


TRAIN
Violence       : 7083
NonViolence    : 9800

VAL
Violence       : 822
NonViolence    : 1152

TEST
Violence       : 843
NonViolence    : 1165

Total Images: 20865


In [13]:
import json

with open("/content/frame_metadata.json") as f:

    metadata = json.load(f)

metadata.keys()

dict_keys(['train', 'val', 'test'])

In [14]:
train_violence = metadata["train"]["Violence"]

frame_counts = list(train_violence.values())

print("Maximum:", max(frame_counts))
print("Minimum:", min(frame_counts))
print("Average:", sum(frame_counts)/len(frame_counts))

Maximum: 355
Minimum: 0
Average: 8.85375


In [15]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMAGE_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

ROOT="/content/Safe5G_Frames"

train_dataset = datasets.ImageFolder(
    ROOT+"/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    ROOT+"/val",
    transform=test_transform
)

test_dataset = datasets.ImageFolder(
    ROOT+"/test",
    transform=test_transform
)

print("Class Mapping:", train_dataset.class_to_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print()

print("Training Images :", len(train_dataset))
print("Validation Images :", len(val_dataset))
print("Testing Images :", len(test_dataset))

Class Mapping: {'NonViolence': 0, 'Violence': 1}

Training Images : 16883
Validation Images : 1974
Testing Images : 2008
